# **1. Worldcover into 1 raster (Raw data)**

Since raw data include many tiles it is imprortant to make only 1 raster

In [2]:
import os
import glob
import rasterio
from rasterio.merge import merge

# --- Paths (your paths) ---
raw_data_dir = r"C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\1 RAW DATA"
tiles_dir    = os.path.join(raw_data_dir, "5 Worldcover")
out_dir      = os.path.join(raw_data_dir, "5 Worldcover 1 raster 10 m")
out_path     = os.path.join(out_dir, "WorldCover_mosaic_10m.tif")

# --- Create output folder ---
os.makedirs(out_dir, exist_ok=True)

# --- Find tiles (adjust extensions if needed) ---
tile_paths = sorted(
    glob.glob(os.path.join(tiles_dir, "*.tif")) +
    glob.glob(os.path.join(tiles_dir, "*.tiff"))
)

if not tile_paths:
    raise FileNotFoundError(f"No .tif/.tiff files found in: {tiles_dir}")

print(f"Found {len(tile_paths)} tiles.")

# --- Open sources, merge, and write output ---
srcs = [rasterio.open(p) for p in tile_paths]

try:
    # Use the first tile as reference for resolution/CRS/dtype/etc.
    ref = srcs[0]
    target_res = ref.res  # should be (10.0, 10.0) for WorldCover 10m

    # Merge (mosaic). res=target_res keeps the 10 m grid size.
    mosaic, out_transform = merge(
        srcs,
        res=target_res,
        nodata=ref.nodata  # keep original nodata if set
    )

    out_meta = ref.meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_transform,
        "compress": "LZW",
        "tiled": True,
        "bigtiff": "IF_SAFER"
    })
    if ref.nodata is not None:
        out_meta["nodata"] = ref.nodata

    with rasterio.open(out_path, "w", **out_meta) as dst:
        dst.write(mosaic)

    print("✅ Saved mosaic to:", out_path)
    print("Output resolution:", target_res)

finally:
    for s in srcs:
        s.close()

Found 6 tiles.
✅ Saved mosaic to: C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\1 RAW DATA\5 Worldcover 1 raster 10 m\WorldCover_mosaic_10m.tif
Output resolution: (8.333333333333333e-05, 8.333333333333333e-05)


# **2. Check resolution and format of all raw data**

In [1]:
########################################################################################
# The code assumes that the rasters are sorted in a specified folder with subfolders
# To check the data quality the code: 
#    1. extracts its CRS (coordinate reference system)
#    2. extracts its resolution (pixel width/height)
#    3. writes and saves a report with summary
# To run the script print in command prompt:
#   1. conda activate Deforestation   # activate your anaconda environment
#   2. cd "C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression"
#   3. python "2 check_data_quality.py"
# The summary will be saved in a specified directory
#######################################################################################





# import packages
import os
import rasterio
from rasterio.errors import RasterioIOError


# dirctory with data
ROOT_DIR = r"C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\1 RAW DATA"
# look for possible rasters format
RASTER_EXTS = (".tif", ".tiff", ".img", ".vrt")
# save summary
REPORT_PATH = os.path.join(os.getcwd(), "raster_report.txt")



def classify_resolution_meters(val_m):
    """function returns a string for a cell size in meters"""
    v = abs(val_m)
    # resolutions are close to
    if abs(v - 30) < 5:
        return "30 m"
    if abs(v - 100) < 20:
        return "100 m"
    if abs(v - 250) < 50:
        return "250 m"
    if abs(v - 500) < 80:
        return "500 m"
    if abs(v - 1000) < 200:
        return "1 km"
    if abs(v - 5000) < 500:
        return "5 km"
    if abs(v - 10000) < 1000:
        return "10 km"

    # return results
    if v < 1000:
        return f"{v:.1f} m"
    else:
        return f"{v/1000:.2f} km"



def pretty_resolution(res, folder_crs):
    """
    function checkes whether resolution units are meters or degrees
    converts degrees to meters if needed
    return a string with spatial resolution in meters or kilometers
    """
    xres, yres = res

    # if CRS unknown, return raw numbers
    if folder_crs is None:
        return f"{xres} x {yres} (units of CRS)"

    # if projected CRS: treat resolution as meters:
    if folder_crs.is_projected:
        x_m = xres
        y_m = yres

    # if geographic CRS (EPSG:4326 or else): 
    # then units are degrees so approximate metres
    elif folder_crs.is_geographic:
        # approximation: 1 degree ~ 111.32 km
        km_per_deg = 111.32
        x_m = xres * km_per_deg * 1000
        y_m = yres * km_per_deg * 1000
    else:
        # return unknown unit type
        return f"{xres} x {yres} (units of CRS)"

    x_str = classify_resolution_meters(x_m)
    y_str = classify_resolution_meters(y_m)
    return f"{x_str} x {y_str}"



def inspect_rasters(root_dir, report_path):
    """functions walks though the data folder entering every subfolder
       looking for a rasters. When found: 
            1) extracts its CRS (coordinate reference system)
            2) extracts its resolution (pixel width/height)
            3) writes and saves a report with summary
    """
    log_lines = []

    for dirpath, dirnames, filenames in os.walk(root_dir):
        # find raster files in given directory
        raster_files = [
            f for f in filenames
            if f.lower().endswith(RASTER_EXTS)
        ]

        if not raster_files:
            continue  # skip folders without rasters

        crs_info = []       # list of (filename, crs)
        res_info = []       # list of (filename, (xres, yres))
        missing_crs = []    # rasters with no CRS
        failed_files = []   # rasters that were not opened
        folder_crs_obj = None  # store one CRS object for this folder

        for fname in raster_files:
            fpath = os.path.join(dirpath, fname)
            try:
                with rasterio.open(fpath) as src:
                    # read CRS and pixel resolution
                    crs = src.crs
                    res = src.res  # (pixel_width, pixel_height)
                    # add them to the lists
                    crs_info.append((fname, crs))
                    res_info.append((fname, res))

                    # if no CRS append to other list
                    if crs is None:
                        missing_crs.append(fname)
                    else:
                        # save one CRS object 
                        if folder_crs_obj is None:
                            folder_crs_obj = crs

            # if failed append to another list
            except RasterioIOError as e:
                failed_files.append((fname, str(e)))

        # build sets of CRS and resolutions present in this folder
        crs_groups = {}
        for fname, crs in crs_info:
            if crs is None:
                continue
            crs_str = crs.to_string()  # like 'EPSG:32634'
            crs_groups.setdefault(crs_str, []).append(fname)

        res_groups = {}
        for fname, res in res_info:
            res_groups.setdefault(res, []).append(fname)

        # -----------------------------------------------
        # Build summary for a current folder 
        # -----------------------------------------------
        log_lines.append("=" * 100)
        log_lines.append(f"Folder: {dirpath}")
        log_lines.append(f"  Rasters found: {len(raster_files)}")
        log_lines.append(f"  Successfully opened: {len(crs_info)}")
        if failed_files:
            log_lines.append("  Could not open these rasters:")
            for fname, msg in failed_files:
                log_lines.append(f"    - {fname} (error: {msg})")

        # CRS check
        if missing_crs:
            log_lines.append("\n  Rasters WITHOUT CRS defined:")
            for fname in missing_crs:
                log_lines.append(f"    - {fname}")
        else:
            log_lines.append("\n  All rasters have a CRS defined.")

        if crs_groups:
            if len(crs_groups) == 1 and not missing_crs:
                crs_str = next(iter(crs_groups.keys()))
                log_lines.append(f"  All rasters share the same CRS: {crs_str}")
            else:
                log_lines.append(f"  Multiple CRS detected ({len(crs_groups)}):")
                for crs_str, files in crs_groups.items():
                    log_lines.append(f"    CRS: {crs_str}")
                    for fname in files:
                        log_lines.append(f"      - {fname}")
        else:
            log_lines.append("  No CRS information could be read from any raster in this folder.")

        # resolution check
        if res_groups:
            if len(res_groups) == 1:
                res = next(iter(res_groups.keys()))
                pretty_res = pretty_resolution(res, folder_crs_obj)
                log_lines.append(f"\n  All rasters share the same resolution: {pretty_res}")
            else:
                log_lines.append("\n  Multiple resolutions detected:")
                for res, files in res_groups.items():
                    pretty_res = pretty_resolution(res, folder_crs_obj)
                    log_lines.append(f"    Resolution {pretty_res}:")
                    for fname in files:
                        log_lines.append(f"      - {fname}")
        else:
            log_lines.append("\n  No resolution information found.")

        # summary if everything is consistent
        if (not missing_crs
            and len(crs_groups) == 1
            and len(res_groups) == 1):
            res = next(iter(res_groups.keys()))
            pretty_res = pretty_resolution(res, folder_crs_obj)
            log_lines.append(
                f"\n  Summary: all rasters in this folder have a CRS"
                # f"resolution {pretty_res}."
            )

        log_lines.append("")  # blank line between folders

    # write report as txt
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("\n".join(log_lines))


if __name__ == "__main__":
    inspect_rasters(ROOT_DIR, REPORT_PATH)
    print(f"Report saved to: {REPORT_PATH}")

Report saved to: c:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\raster_report.txt


# **3. Clip raw data using Kosovo boudaries**

In [4]:
########################################################################################
# This script:
#   1. Loads Kosovo boundary
#   2. Picks a TEMPLATE raster from RAW_ROOT (first one found)
#   3. Crops the template grid to Kosovo (small extent!)
#   4. Rasterizes a Kosovo mask on that cropped grid
#   5. Walks through all RAW rasters (under RAW_ROOT, including subfolders)
#   6. Reprojects each raster to the TEMPLATE Kosovo grid
#   7. Applies the Kosovo mask (outside Kosovo -> NoData, but grid unchanged)
#   8. Saves aligned rasters in CLIPPED_ROOT with SAME CRS, size, transform
#
# To run:
#   1. conda activate Deforestation
#   2. cd "C:\\Users\\oleks\\Desktop\\DEFORESTATION\\2 Logistic regression"
#   3. python "3_clip_rasters_match_kosovo.py"
########################################################################################

import os
import numpy as np
import rasterio
from rasterio.errors import RasterioIOError
from rasterio.warp import reproject, Resampling
from rasterio.features import rasterize
from rasterio.mask import mask as rio_mask
import geopandas as gpd

# --------------------------------------------------------------------------------------
# PATHS (fixed to your setup)
# --------------------------------------------------------------------------------------
BOUNDARY_PATH = (
    r"C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\0 KOSOVO BOUNDARY"
    r"\gadm41_XKO_0.shp"
)

# *** RAW DATA SOURCE ***
RAW_ROOT = r"C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\1 RAW DATA"

# *** OUTPUT FOLDER ***
CLIPPED_ROOT = (
    r"C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA"
)

# supported formats
RASTER_EXTS = (".tif", ".tiff", ".img", ".vrt")

# overwrite existing rasters in CLIPPED_ROOT?
OVERWRITE = True


# --------------------------------------------------------------------------------------
# Helper functions
# --------------------------------------------------------------------------------------
def iter_rasters(root):
    """Yield full paths to all raster files under root (recursively)."""
    for dirpath, dirnames, filenames in os.walk(root):
        for fname in filenames:
            if fname.lower().endswith(RASTER_EXTS):
                yield os.path.join(dirpath, fname)


def choose_template_raster(root):
    """Pick the first raster found under RAW_ROOT as template."""
    for path in iter_rasters(root):
        return path
    raise RuntimeError(f"No rasters found under RAW_ROOT: {root}")


def build_template_and_mask(template_raster_path):
    """
    1. Load Kosovo boundary
    2. Load template raster
    3. Reproject boundary to template CRS
    4. CROP the template grid to Kosovo using rasterio.mask
    5. Rasterize Kosovo mask on the cropped grid
    """
    print("Loading Kosovo boundary shapefile...")
    kosovo = gpd.read_file(BOUNDARY_PATH)
    if kosovo.crs is None:
        raise ValueError("Kosovo boundary shapefile has no CRS defined.")

    print(f"Loading template raster:\n  {template_raster_path}")
    with rasterio.open(template_raster_path) as src:
        template_crs = src.crs

        # Reproject Kosovo boundary to template CRS
        kosovo_template = kosovo.to_crs(template_crs)
        geom = [kosovo_template.geometry.unary_union]

        # *** CROP template to Kosovo extent ***
        print("Cropping template raster to Kosovo extent...")
        data_cropped, transform_cropped = rio_mask(
            src, geom, crop=True
        )  # data_cropped is small (only Kosovo)
        height_cropped, width_cropped = data_cropped.shape[1], data_cropped.shape[2]

    # Optional: free the template data from memory
    del data_cropped

    print("Template grid (cropped to Kosovo):")
    print(f"  CRS     : {template_crs}")
    print(f"  Size    : {width_cropped} x {height_cropped}")
    print(f"  Transform:\n{transform_cropped}\n")

    # Rasterize Kosovo mask on the cropped grid: inside Kosovo = 1, outside = 0
    print("Rasterizing Kosovo mask on cropped template grid...")
    mask_arr = rasterize(
        [(geom[0], 1)],
        out_shape=(height_cropped, width_cropped),
        transform=transform_cropped,
        fill=0,
        dtype="uint8",
    )

    template_info = {
        "crs": template_crs,
        "transform": transform_cropped,
        "width": width_cropped,
        "height": height_cropped,
        "mask": mask_arr,
    }

    return template_info


def process_single_raster(in_path, out_path, template_info):
    """Reproject one raster to the template Kosovo grid and apply Kosovo mask."""
    print(f"[PROCESS] {in_path}")

    with rasterio.open(in_path) as src:
        if src.crs is None:
            print("  Skipping (no CRS defined).")
            return

        dtype = src.dtypes[0]
        src_nodata = src.nodata

        # Define a nodata value if missing
        if src_nodata is None:
            if np.issubdtype(np.dtype(dtype), np.integer):
                src_nodata = 0
            else:
                src_nodata = np.nan

        # Prepare destination metadata
        dst_meta = src.meta.copy()
        dst_meta.update(
            {
                "crs": template_info["crs"],
                "transform": template_info["transform"],
                "width": template_info["width"],
                "height": template_info["height"],
                "nodata": src_nodata,
                "compress": "lzw",
            }
        )

        # Allocate destination array for all bands on the (cropped) Kosovo grid
        dst_array = np.full(
            (src.count, template_info["height"], template_info["width"]),
            src_nodata,
            dtype=dtype,
        )

        # Choose resampling method
        if np.issubdtype(np.dtype(dtype), np.integer):
            resampling = Resampling.nearest  # categorical
        else:
            resampling = Resampling.bilinear  # continuous

        # Reproject each band into the template grid
        for band in range(1, src.count + 1):
            reproject(
                source=rasterio.band(src, band),
                destination=dst_array[band - 1],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=template_info["transform"],
                dst_crs=template_info["crs"],
                src_nodata=src_nodata,
                dst_nodata=src_nodata,
                resampling=resampling,
            )

    # Apply Kosovo mask: outside Kosovo -> NoData
    mask_zero = template_info["mask"] == 0  # bool array, only Kosovo-sized now

    # IMPORTANT: do it band by band to avoid huge temporary arrays
    for i in range(dst_array.shape[0]):
        band = dst_array[i]
        band[mask_zero] = src_nodata
        dst_array[i] = band

    # Ensure output directory exists
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    # Save reprojected & masked raster
    with rasterio.open(out_path, "w", **dst_meta) as dst:
        dst.write(dst_array)

    print(f"  Saved aligned raster to: {out_path}")


# --------------------------------------------------------------------------------------
# Main
# --------------------------------------------------------------------------------------
def main():
    # 1. Choose a template raster from RAW_ROOT
    template_raster_path = choose_template_raster(RAW_ROOT)
    print(f"Template raster selected from RAW_ROOT:\n  {template_raster_path}\n")

    # 2. Build template + Kosovo mask ONCE (cropped to Kosovo!)
    template_info = build_template_and_mask(template_raster_path)

    # 3. Walk through raw rasters and process all
    for dirpath, dirnames, filenames in os.walk(RAW_ROOT):
        raster_files = [f for f in filenames if f.lower().endswith(RASTER_EXTS)]
        if not raster_files:
            continue

        rel_path = os.path.relpath(dirpath, RAW_ROOT)
        out_dir = os.path.join(CLIPPED_ROOT, rel_path)

        for fname in raster_files:
            in_path = os.path.join(dirpath, fname)
            out_path = os.path.join(out_dir, fname)

            if (not OVERWRITE) and os.path.exists(out_path):
                print(f"[SKIP] Already exists and OVERWRITE=False: {out_path}")
                continue

            try:
                process_single_raster(in_path, out_path, template_info)
            except RasterioIOError as e:
                print(f"  Could not open raster: {in_path} (error: {e})")
                continue

    print("\nDone! All rasters are aligned to the Kosovo grid and masked to Kosovo.")
    print(f"Output folder: {CLIPPED_ROOT}")


if __name__ == "__main__":
    main()

Template raster selected from RAW_ROOT:
  C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\1 RAW DATA\1 Forest change 30m\Hansen_GFC-2024-v1.12_datamask_50N_020E.tif

Loading Kosovo boundary shapefile...
Loading template raster:
  C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\1 RAW DATA\1 Forest change 30m\Hansen_GFC-2024-v1.12_datamask_50N_020E.tif


C:\Users\oleks\AppData\Local\Temp\ipykernel_24452\3983154421.py:87: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geom = [kosovo_template.geometry.unary_union]


Cropping template raster to Kosovo extent...
Template grid (cropped to Kosovo):
  CRS     : EPSG:4326
  Size    : 7173 x 5592
  Transform:
| 0.00, 0.00, 20.00|
| 0.00,-0.00, 43.25|
| 0.00, 0.00, 1.00|

Rasterizing Kosovo mask on cropped template grid...
[PROCESS] C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\1 RAW DATA\1 Forest change 30m\Hansen_GFC-2024-v1.12_datamask_50N_020E.tif
  Saved aligned raster to: C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\1 Forest change 30m\Hansen_GFC-2024-v1.12_datamask_50N_020E.tif
[PROCESS] C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\1 RAW DATA\1 Forest change 30m\Hansen_GFC-2024-v1.12_first_50N_020E.tif
  Saved aligned raster to: C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\1 Forest change 30m\Hansen_GFC-2024-v1.12_first_50N_020E.tif
[PROCESS] C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\1 RAW DATA\1 Forest change 30m\Hansen_GFC-2024-v1.12_gain_50N_020

# **4. Check if rasters were clipped properly**

In [5]:
import os
import rasterio
from rasterio.errors import RasterioIOError

# === EDIT THIS TO YOUR CLIPPED DATA FOLDER ===
CLIPPED_ROOT = r"C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA"

# supported formats
RASTER_EXTS = (".tif", ".tiff", ".img", ".vrt")

# tolerance for comparing coordinates (in map units, e.g. meters)
TOL = 1e-6


def iter_rasters(root):
    """Yield full paths to all raster files under root (recursively)."""
    for dirpath, dirnames, filenames in os.walk(root):
        for fname in filenames:
            if fname.lower().endswith(RASTER_EXTS):
                yield os.path.join(dirpath, fname)


def almost_equal(a, b, tol=TOL):
    return abs(a - b) <= tol


def bounds_equal(b1, b2, tol=TOL):
    return (
        almost_equal(b1.left, b2.left, tol) and
        almost_equal(b1.bottom, b2.bottom, tol) and
        almost_equal(b1.right, b2.right, tol) and
        almost_equal(b1.top, b2.top, tol)
    )


def main():
    raster_paths = list(iter_rasters(CLIPPED_ROOT))

    if not raster_paths:
        print(f"No rasters found under: {CLIPPED_ROOT}")
        return

    print(f"Found {len(raster_paths)} rasters under: {CLIPPED_ROOT}")
    print("Using first raster as reference for CRS and bounds.\n")

    # open reference raster
    with rasterio.open(raster_paths[0]) as ref:
        ref_crs = ref.crs
        ref_bounds = ref.bounds
        ref_width, ref_height = ref.width, ref.height
        ref_transform = ref.transform

    print("Reference raster:")
    print(f"  Path    : {raster_paths[0]}")
    print(f"  CRS     : {ref_crs}")
    print(f"  Bounds  : {ref_bounds}")
    print(f"  Size    : {ref_width} x {ref_height}")
    print(f"  Transform:\n{ref_transform}\n")

    mismatches = []

    # check all rasters against reference
    for path in raster_paths[1:]:
        try:
            with rasterio.open(path) as src:
                crs = src.crs
                bounds = src.bounds
                width, height = src.width, src.height
                transform = src.transform
        except RasterioIOError as e:
            print(f"[ERROR] Could not open raster: {path} (error: {e})")
            continue

        reasons = []

        # check CRS
        if crs != ref_crs:
            reasons.append(f"CRS differs (got {crs}, expected {ref_crs})")

        # check bounds
        if not bounds_equal(bounds, ref_bounds, TOL):
            reasons.append(f"Bounds differ (got {bounds}, expected {ref_bounds})")

        # optional: also enforce same size & transform
        if width != ref_width or height != ref_height:
            reasons.append(
                f"Size differs (got {width}x{height}, expected {ref_width}x{ref_height})"
            )

        if transform != ref_transform:
            reasons.append("Affine transform differs")

        if reasons:
            mismatches.append((path, reasons))

    # summary
    if not mismatches:
        print("✅ All rasters have the SAME CRS, bounds, size, and transform.")
    else:
        print("\n❌ Some rasters do NOT match the reference:")
        for path, reasons in mismatches:
            print(f"\nRaster: {path}")
            for r in reasons:
                print(f"  - {r}")


if __name__ == "__main__":
    main()

Found 49 rasters under: C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA
Using first raster as reference for CRS and bounds.

Reference raster:
  Path    : C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\1 Forest change 30m\Hansen_GFC-2024-v1.12_datamask_50N_020E.tif
  CRS     : EPSG:4326
  Bounds  : BoundingBox(left=20.0, bottom=41.84825, right=21.79325, top=43.24625)
  Size    : 7173 x 5592
  Transform:
| 0.00, 0.00, 20.00|
| 0.00,-0.00, 43.25|
| 0.00, 0.00, 1.00|

✅ All rasters have the SAME CRS, bounds, size, and transform.


# **5. Calculate means for selected folders (if applicable, depends of chosen methodology)**

In [6]:
# import packages
import os
import numpy as np
import rasterio

# ============================================================
# PATHS
# ============================================================

CLIPPED_ROOT = r"C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA"

# CHELSA monthly 1 km (input) and mean (output)
CHELSA_ROOT      = os.path.join(CLIPPED_ROOT, "8 CHELSA 1 km")
CHELSA_MEAN_ROOT = os.path.join(CLIPPED_ROOT, "6 CHELSA mean 1 km")

# Population 100m (input) and mean (output)
POP_ROOT        = os.path.join(CLIPPED_ROOT, "4 Population 100m")
POP_MEAN_ROOT   = os.path.join(CLIPPED_ROOT, "7 Population mean 100m")
POP_MEAN_PATH   = os.path.join(POP_MEAN_ROOT, "population_mean_2000_2020.tif")

# CORINE 100m (input) and baseline (output)
#CORINE_ROOT        = os.path.join(CLIPPED_ROOT, "3 Corine land cover 100m")
#CORINE_BASE_ROOT   = os.path.join(CLIPPED_ROOT, "8 Corine baseline 100m")
#CORINE_BASE_PATH   = os.path.join(CORINE_BASE_ROOT, "corine_2000_baseline.tif")

RASTER_EXTS = (".tif", ".tiff")

# Years to use for CHELSA period mean
CHELSA_YEARS = list(range(2000, 2021))  # 2000..2020 inclusive


# ============================================================
# GENERIC HELPERS
# ============================================================

def list_subdirs(path):
    """Return list of immediate subfolder names in given path."""
    return [d for d in os.listdir(path)
            if os.path.isdir(os.path.join(path, d))]


def list_rasters(path):
    """Return list of raster file paths directly in this folder."""
    return [os.path.join(path, f) for f in os.listdir(path)
            if f.lower().endswith(RASTER_EXTS)]


def list_rasters_recursive(root_dir):
    """Return list of raster file paths under root_dir (recursively)."""
    paths = []
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for fname in filenames:
            if fname.lower().endswith(RASTER_EXTS):
                paths.append(os.path.join(dirpath, fname))
    return sorted(paths)


# ============================================================
# 1) CHELSA 2000–2020 MEAN (monthly → period mean)
# ============================================================

def compute_chelsa_period_mean(var_dir, years, out_path):
    """
    Compute mean over all monthly rasters in `var_dir/<year>/` for given years.
    Save one period-mean raster to out_path.
    """
    sum_data = None
    count_data = None
    profile = None
    n_files = 0

    for y in years:
        year_dir = os.path.join(var_dir, str(y))
        if not os.path.exists(year_dir):
            continue

        for fpath in list_rasters(year_dir):
            with rasterio.open(fpath) as src:
                arr = src.read(1, masked=True)

                if sum_data is None:
                    sum_data = np.zeros(arr.shape, dtype=np.float64)
                    count_data = np.zeros(arr.shape, dtype=np.int32)
                    profile = src.profile

                valid = ~arr.mask
                sum_data[valid] += arr.data[valid]
                count_data[valid] += 1
                n_files += 1

    if n_files == 0:
        print(f"  [WARNING] No CHELSA rasters found in {var_dir} for years {years}")
        return

    mean_data = np.zeros_like(sum_data, dtype=np.float32)
    valid = count_data > 0
    mean_data[valid] = (sum_data[valid] / count_data[valid]).astype(np.float32)
    mean_data[~valid] = np.nan

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    out_profile = profile.copy()
    out_profile.update(
        dtype=rasterio.float32,
        count=1,
        nodata=np.nan
    )

    with rasterio.open(out_path, "w", **out_profile) as dst:
        dst.write(mean_data, 1)

    print(f"  [CHELSA mean OK] {var_dir} 2000–2020 from {n_files} rasters -> {out_path}")


def make_chelsa_means():
    """Compute 2000–2020 mean for each CHELSA variable (tasmax, tasmin, cmi, pr)."""
    if not os.path.exists(CHELSA_ROOT):
        print(f"[INFO] CHELSA root {CHELSA_ROOT} does not exist, skipping CHELSA.")
        return

    var_folders = list_subdirs(CHELSA_ROOT)
    print("CHELSA variable folders found:")
    for vf in var_folders:
        print("  -", vf)

    for var_folder in var_folders:
        var_dir = os.path.join(CHELSA_ROOT, var_folder)

        # Extract base_name (e.g. '1 tasmax' -> 'tasmax')
        base_name = var_folder
        parts = var_folder.split()
        if len(parts) > 1 and parts[0].isdigit():
            base_name = parts[1]

        out_dir = os.path.join(CHELSA_MEAN_ROOT, var_folder)
        out_path = os.path.join(out_dir, f"{base_name}_mean_2000_2020.tif")

        if os.path.exists(out_path):
            print(f"[CHELSA SKIP] {out_path} already exists.")
            continue

        print(f"\n[CHELSA PERIOD MEAN] {var_folder} 2000–2020")
        compute_chelsa_period_mean(var_dir, CHELSA_YEARS, out_path)

    print("\nDone. CHELSA 2000–2020 mean rasters are in:")
    print(CHELSA_MEAN_ROOT)


# ============================================================
# 2) POPULATION MEAN 2000–2020
# ============================================================

def make_population_mean():
    """Compute mean over all population rasters in POP_ROOT (per-pixel mean)."""
    rasters = list_rasters_recursive(POP_ROOT)
    if not rasters:
        print(f"[INFO] No population rasters found in {POP_ROOT}, skipping.")
        return

    print("\nPopulation rasters found:")
    for p in rasters:
        print("  ", p)

    sum_data = None
    count_data = None
    profile = None
    n_files = 0

    for fpath in rasters:
        with rasterio.open(fpath) as src:
            arr = src.read(1, masked=True)
            if sum_data is None:
                sum_data = np.zeros(arr.shape, dtype=np.float64)
                count_data = np.zeros(arr.shape, dtype=np.int32)
                profile = src.profile
            else:
                if arr.shape != sum_data.shape:
                    print(f"  [WARNING] Shape mismatch for {fpath}, expected {sum_data.shape}, got {arr.shape}. Skipping.")
                    continue

            valid = ~arr.mask
            sum_data[valid] += arr.data[valid]
            count_data[valid] += 1
            n_files += 1

    if n_files == 0:
        print("[WARNING] No population rasters usable (shape mismatches), skipping population mean.")
        return

    mean_data = np.zeros_like(sum_data, dtype=np.float32)
    valid = count_data > 0
    mean_data[valid] = (sum_data[valid] / count_data[valid]).astype(np.float32)
    mean_data[~valid] = np.nan

    os.makedirs(POP_MEAN_ROOT, exist_ok=True)
    out_profile = profile.copy()
    out_profile.update(
        dtype=rasterio.float32,
        count=1,
        nodata=np.nan
    )

    with rasterio.open(POP_MEAN_PATH, "w", **out_profile) as dst:
        dst.write(mean_data, 1)

    print(f"\n[POPULATION mean OK] from {n_files} rasters -> {POP_MEAN_PATH}")




# ============================================================
# MAIN
# ============================================================

def main():
    make_chelsa_means()
    make_population_mean()
    print("\nAll static summaries for 2000–2020 computed.")


if __name__ == "__main__":
    main()

CHELSA variable folders found:
  - 1 pr
  - 4 spei

[CHELSA PERIOD MEAN] 1 pr 2000–2020
  [CHELSA mean OK] C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\8 CHELSA 1 km\1 pr 2000–2020 from 12 rasters -> C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\6 CHELSA mean 1 km\1 pr\pr_mean_2000_2020.tif

[CHELSA PERIOD MEAN] 4 spei 2000–2020
  [CHELSA mean OK] C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\8 CHELSA 1 km\4 spei 2000–2020 from 12 rasters -> C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\6 CHELSA mean 1 km\4 spei\spei_mean_2000_2020.tif

Done. CHELSA 2000–2020 mean rasters are in:
C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\6 CHELSA mean 1 km

Population rasters found:
   C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\4 Population 100m\kos_ppp_2018.tif

[POPULATION mean OK] from 1 rasters -> C:\Users\oleks\Desktop\DEFOR

## **6. Calculate distances to grasslands, cropland, builtup areas, and permanent water bodies in meters. Harmonization from 10m to 30m is also performed here cause different technique is used**

Euclidean distances in meters are computed on 10m rasters

In [10]:
# --- COPY/PASTE: WorldCover(10m) -> UTM(meters) -> distance rasters (meters) -> aligned to 30m template
# Output folder:  ...\2 CLIPPED DATA\9 distances

from pathlib import Path
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from scipy.ndimage import distance_transform_edt

# ESA WorldCover class codes
CODES = {
    "grassland": 30,
    "cropland": 40,
    "builtup": 50,
    "water": 80,
}

NODATA_DIST = -9999.0  # nodata value for distance rasters (Float32)


def find_first_tif(path_str: str) -> str:
    """Accepts a file path or a directory. If directory, returns the first .tif/.tiff (sorted)."""
    p = Path(path_str)
    if p.is_dir():
        tifs = sorted(list(p.glob("*.tif")) + list(p.glob("*.tiff")))
        if not tifs:
            raise FileNotFoundError(f"No .tif/.tiff found in: {p}")
        return str(tifs[0])
    if not p.exists():
        raise FileNotFoundError(f"Path does not exist: {p}")
    return str(p)


def force_delete(path: Path):
    """Delete output + common sidecars without trying to open them (works even if TIFF is corrupted)."""
    path = Path(path)
    for p in [path, Path(str(path) + ".aux.xml"), Path(str(path) + ".ovr")]:
        if p.exists():
            try:
                p.unlink()
            except PermissionError as e:
                raise PermissionError(
                    f"Can't delete {p}. It is probably open/locked (QGIS/ArcGIS). "
                    f"Close it and restart the kernel/session."
                ) from e


def atomic_write_geotiff(out_path: Path, profile: dict, write_func):
    """
    Write to a temp file first, then rename -> avoids leaving corrupted outputs if the process crashes.
    write_func(dst_dataset) should write the data into the open dataset.
    """
    out_path = Path(out_path)
    tmp_path = out_path.with_suffix(".tmp.tif")
    out_path.parent.mkdir(parents=True, exist_ok=True)

    force_delete(out_path)
    force_delete(tmp_path)

    with rasterio.open(tmp_path, "w", **profile) as dst:
        write_func(dst)

    tmp_path.replace(out_path)


def make_gtiff_profile(
    width: int,
    height: int,
    transform,
    crs,
    dtype,
    nodata,
    count: int = 1,
) -> dict:
    """
    GeoTIFF profile that avoids tiling/block-size headaches.
    (Striped TIFF is the most compatible/reliable on Windows.)
    """
    return {
        "driver": "GTiff",
        "width": width,
        "height": height,
        "count": count,
        "dtype": dtype,
        "crs": crs,
        "transform": transform,
        "nodata": nodata,
        "compress": "DEFLATE",
        "tiled": False,           # <--- important: avoids tile/block errors
        "BIGTIFF": "IF_SAFER",
        "interleave": "band",
    }


def reproject_worldcover_to_utm_10m(src_path: str, dst_path: str, dst_crs="EPSG:32634", dst_res=10) -> str:
    """
    Reproject WorldCover from EPSG:4326 to a metric CRS (UTM) using nearest (categorical).
    Output pixel size is dst_res meters.
    """
    src_path = Path(src_path)
    dst_path = Path(dst_path)

    with rasterio.open(src_path) as src:
        if src.crs is None:
            raise ValueError("Input raster has no CRS.")

        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds, resolution=dst_res
        )

        out_profile = make_gtiff_profile(
            width=width,
            height=height,
            transform=transform,
            crs=dst_crs,
            dtype=src.dtypes[0],
            nodata=src.nodata,
            count=1,
        )

        def _writer(dst):
            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=Resampling.nearest,  # categorical-safe
                src_nodata=src.nodata,
                dst_nodata=src.nodata,
            )

        atomic_write_geotiff(dst_path, out_profile, _writer)

    print(f"Saved reprojected WorldCover: {dst_path}")
    return str(dst_path)


def compute_distances_10m(worldcover_utm_path: str, out_dir: str, class_codes: dict) -> None:
    """
    Compute Euclidean distance (meters) to each class code using scipy distance_transform_edt.
    Input MUST be in a metric CRS (meters), e.g., UTM.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    with rasterio.open(worldcover_utm_path) as src:
        lc = src.read(1)
        src_nodata = src.nodata

        # pixel sizes in meters
        xres = float(src.transform.a)
        yres = float(abs(src.transform.e))

        # valid mask
        valid = np.ones(lc.shape, dtype=bool) if src_nodata is None else (lc != src_nodata)

        for name, code in class_codes.items():
            out_path = out_dir / f"dist_to_{name}_m_10m_utm.tif"
            force_delete(out_path)

            target = (lc == code) & valid

            if not np.any(target):
                dist = np.full(lc.shape, NODATA_DIST, dtype=np.float32)
            else:
                # distance_transform_edt computes distance to nearest ZERO pixel
                edt_in = np.ones(lc.shape, dtype=np.uint8)
                edt_in[target] = 0

                dist = distance_transform_edt(edt_in, sampling=(yres, xres)).astype(np.float32)

                if src_nodata is not None:
                    dist[~valid] = NODATA_DIST

            out_profile = make_gtiff_profile(
                width=src.width,
                height=src.height,
                transform=src.transform,
                crs=src.crs,
                dtype="float32",
                nodata=NODATA_DIST,
                count=1,
            )

            def _writer(dst):
                dst.write(dist, 1)

            atomic_write_geotiff(out_path, out_profile, _writer)
            print(f"Saved 10m distance: {out_path}")


def resample_distance_to_template(src_dist_path: str, template_path: str, out_path: str) -> None:
    """
    Reproject/resample a distance raster to match a template raster (your 30m response grid).
    Uses bilinear (good for continuous distances).
    """
    src_dist_path = Path(src_dist_path)
    template_path = Path(template_path)
    out_path = Path(out_path)

    with rasterio.open(template_path) as tpl:
        tpl_crs = tpl.crs
        tpl_transform = tpl.transform
        tpl_width = tpl.width
        tpl_height = tpl.height

    with rasterio.open(src_dist_path) as src:
        src_data = src.read(1)

        dst_data = np.full((tpl_height, tpl_width), NODATA_DIST, dtype=np.float32)

        reproject(
            source=src_data,
            destination=dst_data,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=tpl_transform,
            dst_crs=tpl_crs,
            resampling=Resampling.bilinear,
            src_nodata=src.nodata,
            dst_nodata=NODATA_DIST,
        )

    out_profile = make_gtiff_profile(
        width=tpl_width,
        height=tpl_height,
        transform=tpl_transform,
        crs=tpl_crs,
        dtype="float32",
        nodata=NODATA_DIST,
        count=1,
    )

    def _writer(dst):
        dst.write(dst_data, 1)

    atomic_write_geotiff(out_path, out_profile, _writer)
    print(f"Saved 30m aligned: {out_path}")


if __name__ == "__main__":

    # -------------------- EDIT THESE TWO INPUTS ONLY --------------------
    WORLDCOVER_INPUT = r"C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\5 Worldcover 1 raster 10 m"
    TEMPLATE_30M = r"C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\1 Forest change 30m\Hansen_GFC-2024-v1.12_treecover2000_50N_020E.tif"
    # -------------------------------------------------------------------

    # Output folder requested by you:
    OUT_FOLDER = Path(r"C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\9 distances")
    OUT_FOLDER.mkdir(parents=True, exist_ok=True)

    # Metric CRS for your area (~20–22E, ~42N) => UTM zone 34N
    DST_CRS = "EPSG:32634"

    # Resolve file paths (if you accidentally pass folders)
    worldcover_src = find_first_tif(WORLDCOVER_INPUT)
    template_30m = find_first_tif(TEMPLATE_30M)

    print("Using WorldCover:", worldcover_src)
    print("Using 30m template:", template_30m)
    print("Output folder:", OUT_FOLDER)

    # 1) Reproject WorldCover to UTM meters @ 10m (nearest)
    worldcover_utm_10m = OUT_FOLDER / "worldcover_utm34N_10m.tif"
    reprojected_wc = reproject_worldcover_to_utm_10m(
        worldcover_src, worldcover_utm_10m, dst_crs=DST_CRS, dst_res=10
    )

    # 2) Compute 10m distance rasters (meters) in UTM
    dist10_dir = OUT_FOLDER / "distances_10m_utm"
    compute_distances_10m(reprojected_wc, dist10_dir, CODES)

    # 3) Resample/align distances to your 30m template grid
    dist30_dir = OUT_FOLDER / "distances_30m_aligned"
    dist30_dir.mkdir(parents=True, exist_ok=True)

    for name in CODES.keys():
        src_dist = dist10_dir / f"dist_to_{name}_m_10m_utm.tif"
        out_30m = dist30_dir / f"dist_to_{name}_m_30m.tif"
        resample_distance_to_template(src_dist, template_30m, out_30m)

    print("\nDONE.")
    print("10m UTM distances:", dist10_dir)
    print("30m aligned distances:", dist30_dir)

Using WorldCover: C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\5 Worldcover 1 raster 10 m\WorldCover_mosaic_10m.tif
Using 30m template: C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\1 Forest change 30m\Hansen_GFC-2024-v1.12_treecover2000_50N_020E.tif
Output folder: C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\9 distances
Saved reprojected WorldCover: C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\9 distances\worldcover_utm34N_10m.tif
Saved 10m distance: C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\9 distances\distances_10m_utm\dist_to_grassland_m_10m_utm.tif
Saved 10m distance: C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\9 distances\distances_10m_utm\dist_to_cropland_m_10m_utm.tif
Saved 10m distance: C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\9 distances\distances_10m_utm\dist_to_builtu

## **7. Calculate slope and elevation**

In [11]:
import os
import numpy as np
import rasterio

TOPO_DIR = r"C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\2 Topography 30m"
CLIPPED_ROOT = os.path.dirname(TOPO_DIR)
OUT_DIR = os.path.join(CLIPPED_ROOT, "9 elevation and slope")
RASTER_EXTS = (".tif", ".tiff")

def list_rasters_recursive(root_dir):
    paths = []
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for fname in filenames:
            if fname.lower().endswith(RASTER_EXTS):
                paths.append(os.path.join(dirpath, fname))
    return sorted(paths)

def main():
    rasters = list_rasters_recursive(TOPO_DIR)
    if not rasters:
        raise FileNotFoundError(f"No rasters found in {TOPO_DIR}")

    dem_path = rasters[0]
    print(f"Using DEM raster:\n  {dem_path}")

    os.makedirs(OUT_DIR, exist_ok=True)
    elev_out = os.path.join(OUT_DIR, "elevation_30m.tif")
    slope_out = os.path.join(OUT_DIR, "slope_deg_30m.tif")

    with rasterio.open(dem_path) as src:
        dem = src.read(1, masked=True)
        profile = src.profile
        transform = src.transform
        crs = src.crs

    # 1) elevation: write DEM as float32
    elev_profile = profile.copy()
    elev_profile.update(dtype=rasterio.float32, count=1)

    with rasterio.open(elev_out, "w", **elev_profile) as dst:
        dst.write(dem.astype(np.float32), 1)

    print(f"[OK] Elevation saved to:\n  {elev_out}")

    # 2) slope: convert to float & set NaN on mask
    dem_float = dem.astype("float32")
    dem_data = dem_float.data.copy()
    dem_data[dem.mask] = np.nan

    # --- convert degree spacing to meters if CRS is geographic ---
    dx_deg = abs(transform.a)
    dy_deg = abs(transform.e)

    if crs is not None and crs.is_geographic:
        # approximate meters per degree at center latitude
        # lat_center from transform (top-left y + half-height * pixel size)
        nrows = dem.shape[0]
        lat_center = transform.f + transform.e * (nrows / 2.0)

        # approximate conversions
        m_per_deg_lat = 111_132.0  # meters per degree latitude
        m_per_deg_lon = 111_320.0 * np.cos(np.deg2rad(lat_center))

        dx = dx_deg * m_per_deg_lon
        dy = dy_deg * m_per_deg_lat
        print(f"CRS is geographic; using dx={dx:.2f} m, dy={dy:.2f} m")
    else:
        # assume projected in meters already
        dx = dx_deg
        dy = dy_deg
        print(f"CRS is projected; using dx={dx:.2f} m, dy={dy:.2f} m")

    # compute gradients in z per meter
    gy, gx = np.gradient(dem_data, dy, dx)
    slope_rad = np.arctan(np.sqrt(gx**2 + gy**2))
    slope_deg = np.degrees(slope_rad).astype("float32")

    slope_deg[dem.mask] = np.nan  # keep outside area as NaN

    slope_profile = profile.copy()
    slope_profile.update(dtype=rasterio.float32, count=1, nodata=np.nan)

    with rasterio.open(slope_out, "w", **slope_profile) as dst:
        dst.write(slope_deg, 1)

    print(f"[OK] Slope saved to:\n  {slope_out}")

if __name__ == "__main__":
    main()

Using DEM raster:
  C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\2 Topography 30m\dtm_elev.lowestmode_gedi.eml_mf_30m_0..0cm_2000..2018_eumap_epsg3035_v0.3_OT.tif
[OK] Elevation saved to:
  C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\9 elevation and slope\elevation_30m.tif
CRS is geographic; using dx=20.50 m, dy=27.78 m
[OK] Slope saved to:
  C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\9 elevation and slope\slope_deg_30m.tif


# **8. Validation of slope and elevation**

In [12]:
import os
import numpy as np
import rasterio

# ============================================================
# PATHS
# ============================================================
CLIPPED_ROOT = r"C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA"

TOPO_DIR = os.path.join(CLIPPED_ROOT, "2 Topography 30m")
OUT_DIR  = os.path.join(CLIPPED_ROOT, "9 elevation and slope")

ELEV_PATH  = os.path.join(OUT_DIR, "elevation_30m.tif")
SLOPE_PATH = os.path.join(OUT_DIR, "slope_deg_30m.tif")

RASTER_EXTS = (".tif", ".tiff")


# ============================================================
# HELPERS
# ============================================================
def list_rasters_recursive(root_dir):
    paths = []
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for fname in filenames:
            if fname.lower().endswith(RASTER_EXTS):
                paths.append(os.path.join(dirpath, fname))
    return sorted(paths)


def describe_raster(path, name=None):
    """Print basic info and stats, ignoring nodata (masked)."""
    if name is None:
        name = os.path.basename(path)
    print(f"\n=== {name} ===")
    with rasterio.open(path) as src:
        arr = src.read(1, masked=True)
        profile = src.profile

    print("  Path :", path)
    print("  Shape:", arr.shape)
    print("  CRS  :", profile["crs"])
    print("  Transform:\n", profile["transform"])

    data = arr.data.astype("float64")
    mask = arr.mask
    valid = data[~mask]

    print(f"  N valid: {valid.size}")
    print(f"  N NaN  : {mask.sum()} ({mask.sum() / data.size:.2%})")

    if valid.size > 0:
        print("  Min   :", np.nanmin(valid))
        print("  Max   :", np.nanmax(valid))
        print("  Mean  :", np.nanmean(valid))
        print("  Std   :", np.nanstd(valid))
    else:
        print("  [WARNING] No valid values!")


def check_elevation_matches_dem():
    """Compare original DEM vs elevation_30m.tif (only over valid pixels)."""
    rasters = list_rasters_recursive(TOPO_DIR)
    if not rasters:
        raise FileNotFoundError(f"No rasters found in {TOPO_DIR}")

    dem_path = rasters[0]
    print(f"\n[DEM] Using original DEM:\n  {dem_path}")

    with rasterio.open(dem_path) as src_dem, rasterio.open(ELEV_PATH) as src_elev:
        dem = src_dem.read(1, masked=True).astype("float64")
        elev = src_elev.read(1, masked=True).astype("float64")

    if dem.shape != elev.shape:
        print("  [WARNING] Shape mismatch:", dem.shape, "vs", elev.shape)

    both_valid = (~dem.mask) & (~elev.mask)
    diff = elev.data[both_valid] - dem.data[both_valid]

    print("\n[CHECK] elevation_30m vs original DEM")
    print("  N compared pixels:", diff.size)
    if diff.size == 0:
        print("  [WARNING] No overlapping valid pixels to compare!")
        return

    print("  Max abs diff:", np.nanmax(np.abs(diff)))
    print("  Mean diff   :", np.nanmean(diff))
    print("  Std diff    :", np.nanstd(diff))


def manual_slope_from_3x3(block, dx_m, dy_m):
    """
    Compute slope in degrees from a 3x3 elevation block using central differences.
    block shape: (3, 3), center at [1,1].
    dx_m, dy_m are pixel spacing in METERS.
    """
    z = block

    dzdx = (z[1, 2] - z[1, 0]) / (2 * dx_m)
    dzdy = (z[2, 1] - z[0, 1]) / (2 * dy_m)

    slope_rad = np.arctan(np.sqrt(dzdx**2 + dzdy**2))
    return np.degrees(slope_rad)


def check_slope_against_manual(n_samples=5, seed=0):
    """Randomly pick pixels and compare slope_deg_30m.tif to manual slope."""
    with rasterio.open(ELEV_PATH) as src_elev, rasterio.open(SLOPE_PATH) as src_slope:
        dem = src_elev.read(1, masked=True).astype("float64")
        slope = src_slope.read(1, masked=True).astype("float64")
        transform = src_elev.transform
        crs = src_elev.crs

    # --- compute grid spacing in METERS (same logic as slope script) ---
    dx_deg = abs(transform.a)
    dy_deg = abs(transform.e)

    nrows = dem.shape[0]
    # center latitude in degrees
    lat_center = transform.f + transform.e * (nrows / 2.0)

    if crs is not None and crs.is_geographic:
        # approximate meters per degree
        m_per_deg_lat = 111_132.0
        m_per_deg_lon = 111_320.0 * np.cos(np.deg2rad(lat_center))
        dx_m = dx_deg * m_per_deg_lon
        dy_m = dy_deg * m_per_deg_lat
    else:
        # assume projected in meters already
        dx_m = dx_deg
        dy_m = dy_deg

    # candidate pixels: interior (for 3x3 window) & valid
    valid = (~dem.mask) & (~slope.mask)
    rows, cols = np.where(valid)
    mask_interior = (rows > 0) & (rows < dem.shape[0] - 1) & (cols > 0) & (cols < dem.shape[1] - 1)
    rows = rows[mask_interior]
    cols = cols[mask_interior]

    if rows.size == 0:
        print("\n[CHECK] slope vs manual: no interior valid pixels to test.")
        return

    rng = np.random.default_rng(seed)
    idx = rng.choice(rows.size, size=min(n_samples, rows.size), replace=False)

    print("\n[CHECK] slope_deg_30m vs manual 3x3 finite-difference slope")
    max_abs_diff = 0.0

    for i in idx:
        r = rows[i]
        c = cols[i]

        block = dem.data[r - 1:r + 2, c - 1:c + 2]
        if np.isnan(block).any():
            continue

        s_manual = manual_slope_from_3x3(block, dx_m, dy_m)
        s_raster = slope.data[r, c]

        diff = s_manual - s_raster
        max_abs_diff = max(max_abs_diff, abs(diff))

        print(f"  Pixel (row={r}, col={c}):")
        print(f"    slope raster : {s_raster:.3f} deg")
        print(f"    slope manual : {s_manual:.3f} deg")
        print(f"    difference   : {diff:.3f} deg")

    print(f"  Max |difference| across samples: {max_abs_diff:.3f} deg")


# ============================================================
# MAIN
# ============================================================
def main():
    # 1) Check files exist
    if not os.path.exists(ELEV_PATH):
        raise FileNotFoundError(f"Elevation raster not found at {ELEV_PATH}")
    if not os.path.exists(SLOPE_PATH):
        raise FileNotFoundError(f"Slope raster not found at {SLOPE_PATH}")

    # 2) Describe elevation & slope rasters (with nodata handled)
    describe_raster(ELEV_PATH, "Elevation 30m")
    describe_raster(SLOPE_PATH, "Slope (deg) 30m")

    # 3) elevation: check identical to original DEM
    check_elevation_matches_dem()

    # 4) slope: basic sanity (min/max, negative, >90)
    with rasterio.open(SLOPE_PATH) as src:
        slope_arr = src.read(1, masked=True)
    data = slope_arr.data.astype("float64")
    mask = slope_arr.mask
    valid = data[~mask]

    if valid.size > 0:
        print("\n[CHECK] Slope basic sanity")
        print("  Min slope   :", valid.min())
        print("  Max slope   :", valid.max())
        print("  >90° values :", np.sum(valid > 90))
        print("  <0° values  :", np.sum(valid < 0))
    else:
        print("\n[CHECK] Slope basic sanity")
        print("  [WARNING] No valid slope values to check!")

    # 5) slope vs manual 3x3 finite difference (using same meter spacing)
    check_slope_against_manual(n_samples=5, seed=0)

    print("\nDone checking elevation and slope.")


if __name__ == "__main__":
    main()


=== Elevation 30m ===
  Path : C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\9 elevation and slope\elevation_30m.tif
  Shape: (5592, 7173)
  CRS  : EPSG:4326
  Transform:
 | 0.00, 0.00, 20.00|
| 0.00,-0.00, 43.25|
| 0.00, 0.00, 1.00|
  N valid: 18994332
  N NaN  : 21117084 (52.65%)
  Min   : 278.98297119140625
  Max   : 2606.525390625
  Mean  : 810.4023721640422
  Std   : 398.28813362981515

=== Slope (deg) 30m ===
  Path : C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\9 elevation and slope\slope_deg_30m.tif
  Shape: (5592, 7173)
  CRS  : EPSG:4326
  Transform:
 | 0.00, 0.00, 20.00|
| 0.00,-0.00, 43.25|
| 0.00, 0.00, 1.00|
  N valid: 18970408
  N NaN  : 21141008 (52.71%)
  Min   : 0.0
  Max   : 71.6026611328125
  Mean  : 11.225358843951703
  Std   : 8.777693439915843

[DEM] Using original DEM:
  C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\2 Topography 30m\dtm_elev.lowestmode_gedi.eml_mf_30m_0..0cm_20

# **9. Harmonization from big size to 30m**

In [13]:
import os
import shutil
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.errors import RasterioIOError

# =====================================================================
# PATHS
# =====================================================================

# main input folder (all clipped data)
CLIPPED_ROOT = r"C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA"

# output folder for harmonized 30m data
HARMONIZED_ROOT = r"C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\3 HARMONIZED DATA"

# top-level subfolders in 2 CLIPPED DATA that should NOT be harmonized
EXCLUDE_TOP_LEVELS = {
    "2 Topography 30m",
    "3 Corine land cover 100m",
    "4 Population 100m",
    "5 Worldcover",
    "5 Worldcover 1 raster 10 m",
    "8 CHELSA 1 km",
    "9 distances",
  # monthly 1 km products
}

# raster formats
RASTER_EXTS = (".tif", ".tiff")


# =====================================================================
# HELPERS
# =====================================================================

def list_rasters(root_dir):
    """Yield paths to rasters using recursion."""
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for fname in filenames:
            if fname.lower().endswith(RASTER_EXTS):
                yield os.path.join(dirpath, fname)


def find_30m_reference_raster(root_dir):
    """
    Find the first raster in a top-level folder whose name contains '30m'.
    This defines the 30m grid (CRS, extent, transform).
    """
    for dirpath, dirnames, filenames in os.walk(root_dir):
        rel = os.path.relpath(dirpath, root_dir)
        if rel == ".":
            continue
        top_level = rel.split(os.sep)[0]

        if "30m" not in top_level:
            continue

        for fname in filenames:
            if fname.lower().endswith(RASTER_EXTS):
                return os.path.join(dirpath, fname)

    raise FileNotFoundError(
        f"No 30m raster found under:\n  {root_dir}\n"
        "Make sure at least one top-level folder name contains '30m' and has a .tif inside."
    )


def make_template_from_reference(ref_path):
    """
    Get CRS, transform, width, height, and a base profile from a 30m raster.
    The harmonized grid will match this (same resolution and extent).
    """
    with rasterio.open(ref_path) as src:
        profile = src.profile.copy()

    # We'll override dtype/count/nodata per-dataset later,
    # but keep driver, crs, transform, width, height.
    base_profile = {
        "driver": profile.get("driver", "GTiff"),
        "crs": profile["crs"],
        "transform": profile["transform"],
        "width": profile["width"],
        "height": profile["height"],
    }
    return profile["crs"], profile["transform"], profile["width"], profile["height"], base_profile


def copy_file(src, dst):
    """Copy any file, creating parent folder if needed."""
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)


def resample_to_30m(src_path, tmpl_info, dst_path):
    """
    Resample raster to 30 m grid.

    Heuristic:
      - float dtype   -> continuous, bilinear interpolation
      - integer dtype -> categorical/counts, nearest neighbour
    """
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)

    tmpl_crs, tmpl_transform, tmpl_width, tmpl_height, tmpl_profile = tmpl_info

    with rasterio.open(src_path) as src:
        src_data = src.read(1)  # plain array
        src_profile = src.profile
        src_dtype = src_profile["dtype"]
        src_crs = src_profile["crs"]
        src_transform = src_profile["transform"]
        src_nodata = src_profile.get("nodata", None)

    # Decide resampling + output dtype/nodata
    if np.issubdtype(np.dtype(src_dtype), np.floating):
        # Continuous data (CHELSA means, distances, etc.)
        resampling = Resampling.bilinear
        out_dtype = rasterio.float32
        out_nodata = np.nan
    else:
        # Integer data (e.g. classes/counts)
        resampling = Resampling.nearest
        out_dtype = src_dtype
        # Use source nodata if present, else 0 as a safe dummy
        out_nodata = src_nodata if src_nodata is not None else 0

    # Allocate destination array filled with nodata
    dst_data = np.full((tmpl_height, tmpl_width), out_nodata, dtype=out_dtype)

    reproject(
        source=src_data,
        destination=dst_data,
        src_transform=src_transform,
        src_crs=src_crs,
        src_nodata=src_nodata,
        dst_transform=tmpl_transform,
        dst_crs=tmpl_crs,
        dst_nodata=out_nodata,
        resampling=resampling,
    )

    out_profile = tmpl_profile.copy()
    out_profile.update(
        dtype=out_dtype,
        count=1,
        nodata=out_nodata,
    )

    with rasterio.open(dst_path, "w", **out_profile) as dst:
        dst.write(dst_data, 1)


# =====================================================================
# MAIN
# =====================================================================

def main():
    # 0) Build 30 m template from first 30 m raster
    ref_raster = find_30m_reference_raster(CLIPPED_ROOT)
    print(f"Using 30 m reference raster to define harmonized grid:\n  {ref_raster}")
    tmpl_info = make_template_from_reference(ref_raster)

    # 1) Walk through all files under CLIPPED_ROOT
    for dirpath, dirnames, filenames in os.walk(CLIPPED_ROOT):
        rel_dir = os.path.relpath(dirpath, CLIPPED_ROOT)

        if rel_dir != ".":
            top_level = rel_dir.split(os.sep)[0]
        else:
            top_level = ""

        # ---- skip selected top-level folders entirely ----
        if top_level in EXCLUDE_TOP_LEVELS:
            dirnames[:] = []  # don’t descend further
            continue

        # For info: is this directory under an original 30m folder?
        is_30m_folder = "30m" in top_level

        for fname in filenames:
            src = os.path.join(dirpath, fname)

            # Build relative path (from CLIPPED_ROOT) and adjust top-level folder name to include '30m'
            rel_path = os.path.relpath(src, CLIPPED_ROOT)
            parts = rel_path.split(os.sep)

            if len(parts) > 0:
                orig_top = parts[0]
                if "30m" in orig_top:
                    new_top = orig_top  # already has 30m
                else:
                    new_top = orig_top + " 30m"  # append 30m for harmonized version
                parts[0] = new_top

            rel_path_30m = os.path.join(*parts)
            dst = os.path.join(HARMONIZED_ROOT, rel_path_30m)

            # Non-raster files: always copy (with renamed top-level folder)
            if not fname.lower().endswith(RASTER_EXTS):
                copy_file(src, dst)
                continue

            # Rasters in original 30m folders (except excluded ones): copy as-is
            if is_30m_folder:
                copy_file(src, dst)
                print(f"[COPY 30m] {src} -> {dst}")
                continue

            # Rasters in other folders: resample to 30 m grid
            if os.path.exists(dst):
                print(f"[SKIP] {dst} (already exists)")
                continue

            print(f"[RESAMPLE →30m] {src} -> {dst}")
            try:
                resample_to_30m(src, tmpl_info, dst)
            except RasterioIOError as e:
                print(f"  ⚠ Could not process {src}: {e}")

    print("\nDone. All eligible data (except 2 Topography 30m, 3 Corine land cover 100m, 4 Population 100m, 5 CHELSA 1 km) are now under:")
    print(HARMONIZED_ROOT)
    print("All top-level folders in 3 HARMONIZED DATA have '30m' in their names.")


if __name__ == "__main__":
    main()

Using 30 m reference raster to define harmonized grid:
  C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\1 Forest change 30m\Hansen_GFC-2024-v1.12_datamask_50N_020E.tif
[COPY 30m] C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\1 Forest change 30m\Hansen_GFC-2024-v1.12_datamask_50N_020E.tif -> C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\3 HARMONIZED DATA\1 Forest change 30m\Hansen_GFC-2024-v1.12_datamask_50N_020E.tif
[COPY 30m] C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\1 Forest change 30m\Hansen_GFC-2024-v1.12_first_50N_020E.tif -> C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\3 HARMONIZED DATA\1 Forest change 30m\Hansen_GFC-2024-v1.12_first_50N_020E.tif
[COPY 30m] C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\1 Forest change 30m\Hansen_GFC-2024-v1.12_gain_50N_020E.tif -> C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\3 HARMONIZED DATA\

# **10. Take vectors with roads, clip data, build a raster with distances 30m**

In [14]:
from pathlib import Path
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.features import rasterize
from scipy.ndimage import distance_transform_edt
import geopandas as gpd

# --------------------------------------------------------------------------------------
# EDIT THESE PATHS
# --------------------------------------------------------------------------------------
PBF_PATH = r"C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\1 RAW DATA\9 Roads\kosovo-251218.osm.pbf"

# MUST be UTM meters at 10m resolution (EPSG:32634) and (ideally) already clipped to Kosovo
TEMPLATE_10M_UTM = r"C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\9 distances\worldcover_utm34N_10m.tif"

# This should be your *already clipped/aligned* 30m template raster (same grid as your other clipped rasters)
TEMPLATE_30M = r"C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\2 CLIPPED DATA\1 Forest change 30m\Hansen_GFC-2024-v1.12_treecover2000_50N_020E.tif"

# Kosovo boundary used by your clipping script (to mask outside Kosovo)
BOUNDARY_PATH = (
    r"C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\0 KOSOVO BOUNDARY"
    r"\gadm41_XKO_0.shp"
)

# Output folder requested
OUT_FOLDER = Path(r"C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\3 HARMONIZED DATA\10 road")
OUT_RASTER = OUT_FOLDER / "dist_to_any_road_m_30m.tif"
# --------------------------------------------------------------------------------------

NODATA_DIST = -9999.0


def force_delete(path: Path):
    """Delete output + sidecars without opening (works even if a TIFF is corrupted)."""
    path = Path(path)
    for p in [path, Path(str(path) + ".aux.xml"), Path(str(path) + ".ovr")]:
        if p.exists():
            try:
                p.unlink()
            except PermissionError as e:
                raise PermissionError(
                    f"Can't delete {p}. Probably open/locked (QGIS/ArcGIS). Close and restart kernel."
                ) from e


def atomic_write_geotiff(out_path: Path, profile: dict, write_func):
    """Write to temp then rename to avoid half-written/corrupt GeoTIFFs."""
    out_path = Path(out_path)
    tmp_path = out_path.with_suffix(".tmp.tif")
    out_path.parent.mkdir(parents=True, exist_ok=True)

    force_delete(out_path)
    force_delete(tmp_path)

    with rasterio.open(tmp_path, "w", **profile) as dst:
        write_func(dst)

    tmp_path.replace(out_path)


def make_striped_gtiff_profile(width, height, transform, crs, dtype, nodata, count=1):
    """Striped TIFF is very reliable on Windows (avoids tile/block errors)."""
    return {
        "driver": "GTiff",
        "width": width,
        "height": height,
        "count": count,
        "dtype": dtype,
        "crs": crs,
        "transform": transform,
        "nodata": nodata,
        "compress": "LZW",   # closer to your clipping script style
        "tiled": False,
        "BIGTIFF": "IF_SAFER",
        "interleave": "band",
    }


def kosovo_mask_on_template(template_path: str, boundary_path: str) -> np.ndarray:
    """
    Rasterize Kosovo boundary onto the template grid:
    inside Kosovo = 1, outside = 0.
    """
    kosovo = gpd.read_file(boundary_path)
    if kosovo.crs is None:
        raise ValueError("Kosovo boundary shapefile has no CRS defined.")

    with rasterio.open(template_path) as tpl:
        tpl_crs = tpl.crs
        tpl_transform = tpl.transform
        tpl_h, tpl_w = tpl.height, tpl.width

    kosovo_tpl = kosovo.to_crs(tpl_crs)
    geom = kosovo_tpl.geometry.unary_union

    mask_arr = rasterize(
        [(geom, 1)],
        out_shape=(tpl_h, tpl_w),
        transform=tpl_transform,
        fill=0,
        dtype="uint8",
        all_touched=True,
    )
    return mask_arr


def rasterize_roads_mask_10m(roads_gdf, template_10m_path: str, buffer_m: float = 0.0) -> np.ndarray:
    """Rasterize roads to match the 10m UTM template grid. Roads=1, background=0."""
    with rasterio.open(template_10m_path) as tpl:
        shape = (tpl.height, tpl.width)
        transform = tpl.transform
        crs = tpl.crs

    gdf = roads_gdf
    if gdf.crs is None:
        raise ValueError("Road GeoDataFrame has no CRS.")
    if str(gdf.crs) != str(crs):
        gdf = gdf.to_crs(crs)

    if buffer_m and buffer_m > 0:
        gdf = gdf.copy()
        gdf["geometry"] = gdf.geometry.buffer(buffer_m)

    geoms = [(geom, 1) for geom in gdf.geometry if geom is not None and not geom.is_empty]
    if len(geoms) == 0:
        raise RuntimeError("No valid road geometries to rasterize.")

    mask_arr = rasterize(
        geoms,
        out_shape=shape,
        transform=transform,
        fill=0,
        dtype=np.uint8,
        all_touched=True,
    )
    return mask_arr


def distance_edt_meters_from_mask(mask_10m: np.ndarray, template_10m_path: str) -> np.ndarray:
    """Compute Euclidean distance (meters) to nearest road pixel (mask==1) on the 10m UTM grid."""
    with rasterio.open(template_10m_path) as tpl:
        transform = tpl.transform
        height, width = tpl.height, tpl.width

    xres = float(transform.a)
    yres = float(abs(transform.e))

    edt_in = np.ones((height, width), dtype=np.uint8)
    edt_in[mask_10m == 1] = 0

    dist = distance_transform_edt(edt_in, sampling=(yres, xres)).astype(np.float32)
    return dist


def reproject_to_template(src_arr: np.ndarray, src_template_path: str, dst_template_path: str) -> np.ndarray:
    """Reproject/resample array (with src template georef) to match dst template grid."""
    with rasterio.open(src_template_path) as src_tpl:
        src_crs = src_tpl.crs
        src_transform = src_tpl.transform

    with rasterio.open(dst_template_path) as dst_tpl:
        dst_crs = dst_tpl.crs
        dst_transform = dst_tpl.transform
        dst_h, dst_w = dst_tpl.height, dst_tpl.width

    dst_arr = np.full((dst_h, dst_w), NODATA_DIST, dtype=np.float32)

    reproject(
        source=src_arr,
        destination=dst_arr,
        src_transform=src_transform,
        src_crs=src_crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.bilinear,
        src_nodata=None,            # distances have no nodata on the 10m grid
        dst_nodata=NODATA_DIST,
    )
    return dst_arr


def main():
    OUT_FOLDER.mkdir(parents=True, exist_ok=True)

    # --- 1) Read roads from PBF ---
    from pyrosm import OSM
    osm = OSM(PBF_PATH)

    # "driving" network = roads usable for cars (usually best for accessibility)
    roads = osm.get_network(network_type="driving")
    if roads is None or len(roads) == 0:
        raise RuntimeError("No driving roads were extracted from the PBF.")

    # --- 2) Rasterize roads on 10m UTM template and compute distance (meters) ---
    # buffer_m: 0 = centerline; try 5–10 if you fear missing thin lines
    buffer_m = 0.0
    mask_10m = rasterize_roads_mask_10m(roads, TEMPLATE_10M_UTM, buffer_m=buffer_m)
    dist_10m_m = distance_edt_meters_from_mask(mask_10m, TEMPLATE_10M_UTM)

    # --- 3) Reproject/resample distances to the (already clipped) 30m template grid ---
    dist_30m_m = reproject_to_template(dist_10m_m, TEMPLATE_10M_UTM, TEMPLATE_30M)

    # --- 4) Apply Kosovo boundary mask on the 30m grid (same logic as your clipping workflow) ---
    kz_mask_30m = kosovo_mask_on_template(TEMPLATE_30M, BOUNDARY_PATH)
    dist_30m_m[kz_mask_30m == 0] = NODATA_DIST

    # --- 5) Write ONE output raster aligned to TEMPLATE_30M grid ---
    with rasterio.open(TEMPLATE_30M) as tpl30:
        prof = make_striped_gtiff_profile(
            width=tpl30.width,
            height=tpl30.height,
            transform=tpl30.transform,
            crs=tpl30.crs,
            dtype="float32",
            nodata=NODATA_DIST,
            count=1,
        )

    def _writer(dst):
        dst.write(dist_30m_m.astype(np.float32), 1)

    atomic_write_geotiff(OUT_RASTER, prof, _writer)

    print("\nDONE. Wrote a single 30m distance-to-any-road raster aligned to your clipped grid:")
    print(" -", OUT_RASTER)


if __name__ == "__main__":
    main()

c:\Users\oleks\miniconda3\envs\Deforestation\lib\site-packages\pyrosm\networks.py:37: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  edges, nodes = prepare_geodataframe(
C:\Users\oleks\AppData\Local\Temp\ipykernel_24452\2929781674.py:95: Depr


DONE. Wrote a single 30m distance-to-any-road raster aligned to your clipped grid:
 - C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\3 HARMONIZED DATA\10 road\dist_to_any_road_m_30m.tif


# **VISUALIZATION OF RASTERS**

In [2]:
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt

# Use raw string to avoid issues with backslashes and \t
raster_path = r"C:\Users\oleks\Desktop\DEFORESTATION\2 Logistic regression\1 RAW DATA\5 Worldcover\ESA_WorldCover_10m_2020_v100_N42E021_Map.tif"

# Open and plot
with rasterio.open(raster_path) as src:
    fig, ax = plt.subplots(figsize=(8, 6))
    show(src, ax=ax)  # you can add cmap="viridis" or similar if you like
    ax.set_title("CHELSA tasmax mean 2000–2020")
    plt.show()

MemoryError: Unable to allocate 38.6 GiB for an array with shape (36000, 36000, 4) and data type float64

<Figure size 800x600 with 1 Axes>